In [208]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5] 
q = df[:, 6] 

p[78:] -= 4
q[78:] -= 4

err_p = df[:, 8]
err_q =  df[:, 7]



p = np.delete(p, [0, 62, 88] )
q = np.delete(q, [0, 62, 88]  )
err_p = np.delete(err_p, [0, 62, 88] )
err_q = np.delete(err_q, [0, 62, 88] )



#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7824.12
NDf                       =          110
Edm                       =  2.09454e-07
NCalls                    =          138
A                         =      9.34675   +/-   0.193526    
B                         =       2.8731   +/-   0.143333    
C                         =       196.85   +/-   0.0864761   

Parametri stimati:
A = 9.346750816564576 ± 0.1935258213248982
B = 2.8730986103345635 ± 0.1433325558963419
C = 196.8502103582011 ± 0.08647608502192039
Chi2 = 7824.1234287592315


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


## Fit separati a p = 400



In [209]:

p_cut = 400

model_fit_1 = ROOT.TF1('model_1', model, min(p), p_cut, 3)      # parte con p < 400
model_fit_1.SetParameters(par[0], par[1], par[2])
model_fit_1.SetParNames("A", "B", "C")
model_fit_1.SetLineColor(ROOT.kRed + 1)
model_fit_1.SetLineWidth(2)

model_fit_2 = ROOT.TF1('model_2', model, p_cut, max(p), 3)      # parte con p > 400
model_fit_2.SetParameters(par[0], par[1], par[2])
model_fit_2.SetParNames("A", "B", "C")
model_fit_2.SetLineColor(ROOT.kGreen + 2)
model_fit_2.SetLineWidth(2)

gr_split = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr_split.SetTitle("Fit separati; p; q")
gr_split.SetMarkerStyle(20)
gr_split.SetMarkerSize(0.7)
gr_split.SetMarkerColor(ROOT.kAzure + 2)
gr_split.SetLineColor(ROOT.kAzure + 3)
gr_split.SetLineWidth(1)

fit_res_1 = gr_split.Fit(model_fit_1, "SRN")
fit_res_2 = gr_split.Fit(model_fit_2, "SRN")

par_1 = model_fit_1.GetParameters()
par_2 = model_fit_2.GetParameters()
errors_1 = model_fit_1.GetParErrors()
errors_2 = model_fit_2.GetParErrors()
chi2_1, ndf_1 = fit_res_1.Chi2(), fit_res_1.Ndf()
chi2_2, ndf_2 = fit_res_2.Chi2(), fit_res_2.Ndf()


print(f"Fit 1 (p < {p_cut}):")
print(f"A = {par_1[0]} ± {errors_1[0]}")
print(f"B = {par_1[1]} ± {errors_1[1]}")
print(f"C = {par_1[2]} ± {errors_1[2]}")
print(f"Chi2 = {chi2_1}   ndf = {ndf_1}")
print()
print(f"Fit 2 (p > {p_cut}):")
print(f"A = {par_2[0]} ± {errors_2[0]}")
print(f"B = {par_2[1]} ± {errors_2[1]}")
print(f"C = {par_2[2]} ± {errors_2[2]}")
print(f"Chi2 = {chi2_2}   ndf = {ndf_2}")

print()
for i, nome in enumerate(["A", "B", "C"]):
    diff = np.abs(par_1[i] - par_2[i])
    sigma = np.sqrt(errors_1[i]**2 + errors_2[i]**2)
    if diff < 3*sigma:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> compatibili")
    else:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> NON compatibili")

chi2_tot = model_fit.GetChisquare()      
ndf_tot = model_fit.GetNDF()
chi2_split = chi2_1 + chi2_2            
ndf_split = ndf_1 + ndf_2
print()
print(f"Fit unico    : chi2 = {chi2_tot:.1f}  ndf = {ndf_tot}  chi2/ndf = {chi2_tot/ndf_tot:.1f}")
print(f"Due fit somma: chi2 = {chi2_split:.1f}  ndf = {ndf_split}  chi2/ndf = {chi2_split/ndf_split:.1f}")

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1809.71
NDf                       =           74
Edm                       =  3.36461e-07
NCalls                    =          113
A                         =     -19.2463   +/-   1.92261     
B                         =     -51.2153   +/-   2.88752     
C                         =       216.28   +/-   1.16602     
Fit 1 (p < 400):
A = -19.24634350697729 ± 1.922608568827007
B = -51.21528132921077 ± 2.8875180569108707
C = 216.27993672373285 ± 1.1660169340736948
Chi2 = 1809.711818493685   ndf = 74

Fit 2 (p > 400):
A = -8.457627112922824 ± 0.5457107173508737
B = -6.054883427563692 ± 0.28248581074985674
C = 203.14991703166558 ± 0.19333007243117403
Chi2 = 1956.283198208839   ndf = 33

A: differenza = 10.789 (5.4 sigma) -> NON compatibili
B: differenza = 45.160 (15.6 sigma) -> NON compatibili
C: differenza = 13.130 (11.1 sigma) -> NON compatibili

Fit unico    : chi2 = 7824.1  ndf = 110  

In [210]:
c_split = ROOT.TCanvas("c_split", "Fit separati", 800, 600)
c_split.SetGrid()

gr_split.Draw("AP")
model_fit_1.Draw("same")
model_fit_2.Draw("same")

leg_split = ROOT.TLegend(0.40, 0.72, 0.89, 0.89)
leg_split.AddEntry(gr_split, "Dati", "pe")
leg_split.AddEntry(model_fit_1, f"Fit p < {p_cut}: A = {par_1[0]:.1f} #pm {errors_1[0]:.1f}, #chi^{{2}}/ndf = {chi2_1:.0f}/{ndf_1}", "l")
leg_split.AddEntry(model_fit_2, f"Fit p > {p_cut}: A = {par_2[0]:.1f} #pm {errors_2[0]:.1f}, #chi^{{2}}/ndf = {chi2_2:.0f}/{ndf_2}", "l")
leg_split.Draw()

c_split.Draw()
c_split.SaveAs("fit_root_split.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_split
Info in <TCanvas::Print>: png file fit_root_split.png has been created


In [211]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A NON è compatibile con 0
---
B è compatibile con 0


In [212]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff



#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 40.76688351166224 ± 6.253452292092995
#mu = 0.4265231025495916 ± 0.26983128058420697
#sigma = 2.3451521634988026 ± 0.2500748292660979
Chi2 = 29.57569057315509


-2.8113867532316412
0
-1.1440507836861202
1
-1.6703034399898609
2
-1.5215108101390549
3
-1.3923295827712787
4
-1.2210661794234934
5
-1.556133771696716
6
-1.3009024167249865
7
-1.335721205152874
8
-1.5761023504337004
9
-0.9684023459366671
10
-1.386523639582662
11
-1.2364532683850256
12
-1.3606918693324856
13
-5.261917633372351
60
-4.138926982070987
72
-9.12449166608087
73
-27.800156122818795
74
-37.52535968998735
75
-37.35193988926945
76
-19.066297199060617
77
-7.631033744833744
78
-9.950209483464524
79
-8.784633827488397
83
-1.6091038786409055
90
-10.750931801132225
91
-7.902167242918551
98
-4.736102363162169
103
-8.321953220776876
106
-10.35228236709599
107
-5.3490487345611415
108
-6.978481806251743
110
-1.7648915859228285
111
-8.546332550836315
112
*************************************

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [213]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu è compatibile con 0
---
sigma NON è compatibile con 1


In [214]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[-9.64423616e+01 -5.42101798e+01 -7.38056268e+01 -6.19364506e+01
 -5.50311074e+01 -4.71728421e+01 -4.71385079e+01 -3.71337072e+01
 -4.07699100e+01 -3.76902782e+01 -2.20264096e+01 -3.34892034e+01
 -2.55940940e+01 -2.81394039e+01 -1.18888271e+01 -9.06035256e+00
 -8.49351826e+00 -1.02767732e+01 -7.33052147e+00 -9.82309939e+00
 -4.15215157e+00 -4.84032103e+00  5.42822796e-01 -2.77650552e+00
 -3.27346288e+00 -4.74671384e+00 -5.39122591e+00  4.52660740e-02
  1.91436253e+00 -1.01693775e-01  5.43038938e+00  3.02760098e+00
  3.16915604e+00  4.08371667e+00  5.55941364e+00  6.27915552e+00
  8.43160720e+00  8.52511501e+00  1.12586893e+01  8.63946992e+00
  1.10566632e+01  1.03077765e+01  5.45801021e+00  3.83848942e+00
  9.07875855e+00  6.67930414e+00  8.13633615e+00  1.04147048e+01
  8.57079061e+00  9.08846268e+00  1.14676319e+01  7.76693241e+00
  1.03873151e+01  9.42819415e+00  1.13420956e+01  1.08322075e+01
  1.14409392e+01  8.64474517e+00  9.92082144e+00  1.07799385e+01
 -2.02827509e+00  1.17339

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [215]:
mask_1 = p < p_cut
p_1 = p[mask_1]
q_1 = q[mask_1]
err_p_1 = err_p[mask_1]
err_q_1 = err_q[mask_1]

q_attesi_1 = np.array([model_fit_1.Eval(x) for x in p_1])
err_qeff_1 = np.sqrt(err_q_1**2 + ((model_derivative(p_1, par_1)) * err_p_1)**2)

residuals_1 = q_1 - q_attesi_1
residuals_norm_1 = residuals_1 / err_qeff_1

gaussian_fit_1 = ROOT.TF1('gaussian_1', 'gaus', min(residuals_norm_1), max(residuals_norm_1), 3)
gaussian_fit_1.SetLineColor(ROOT.kRed + 1)
gaussian_fit_1.SetLineWidth(3)

hist_1 = ROOT.TH1F("hist_1", "Istogramma residui norm. (p < 400); (q - q_{att})/err; conteggi", 15, min(residuals_norm_1), max(residuals_norm_1))
for val in residuals_norm_1:
    hist_1.Fill(val)
hist_1.SetLineColor(ROOT.kAzure + 3)
hist_1.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)

c_hist_1 = ROOT.TCanvas("c_hist_1", "Hist Residui 1", 800, 600)
hist_1.Fit(gaussian_fit_1, "S")
hist_1.Draw()
c_hist_1.Draw()

line_fit_1 = ROOT.TF1('line_1', line, min(p_1), max(p_1), 2)
line_fit_1.SetParameters(0, 0) # Inizializziamo a 0
line_fit_1.SetLineColor(ROOT.kRed + 1)

gr_res_1 = ROOT.TGraphErrors(len(p_1), p_1, residuals_1, np.zeros_like(p_1), err_qeff_1)
gr_res_1.SetTitle("Residui in funzione di p (p < 400); p; q - q_{att}")
gr_res_1.SetMarkerStyle(20)
gr_res_1.SetMarkerColor(ROOT.kAzure + 2)
gr_res_1.SetLineColor(ROOT.kAzure + 3)

c_res_1 = ROOT.TCanvas("c_res_1", "Residui vs p 1", 800, 600)
c_res_1.SetGrid()
gr_res_1.Fit(line_fit_1, "S")
gr_res_1.Draw("AP")
c_res_1.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: hist_1 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_hist_1


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      11.9976
NDf                       =           10
Edm                       =  1.25173e-06
NCalls                    =          129
Constant                  =      60.7518   +/-   8.9395      
Mean                      =      1.35012   +/-   0.17394     
Sigma                     =     0.982329   +/-   0.0867315    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1807.43
NDf                       =           75
Edm                       =   5.0198e-20
NCalls                    =           47
p0                        =  -0.00367451   +/-   0.00243037  
p1                        =      1.36294   +/-   0.90316     


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_res_1


In [216]:
mask_2 = p >= p_cut
p_2 = p[mask_2]
q_2 = q[mask_2]
err_p_2 = err_p[mask_2]
err_q_2 = err_q[mask_2]

q_attesi_2 = np.array([model_fit_2.Eval(x) for x in p_2])
err_qeff_2 = np.sqrt(err_q_2**2 + ((model_derivative(p_2, par_2)) * err_p_2)**2)

residuals_2 = q_2 - q_attesi_2
residuals_norm_2 = residuals_2 / err_qeff_2

gaussian_fit_2 = ROOT.TF1('gaussian_2', 'gaus', min(residuals_norm_2), max(residuals_norm_2), 3)
gaussian_fit_2.SetLineColor(ROOT.kGreen + 2)
gaussian_fit_2.SetLineWidth(3)

hist_2 = ROOT.TH1F("hist_2", "Istogramma residui norm. (p > 400); (q - q_{att})/err; conteggi", 15, min(residuals_norm_2), max(residuals_norm_2))
for val in residuals_norm_2:
    hist_2.Fill(val)
hist_2.SetLineColor(ROOT.kAzure + 3)
hist_2.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)

c_hist_2 = ROOT.TCanvas("c_hist_2", "Hist Residui 2", 800, 600)
hist_2.Fit(gaussian_fit_2, "S")
hist_2.Draw()
c_hist_2.Draw()

line_fit_2 = ROOT.TF1('line_2', line, min(p_2), max(p_2), 2)
line_fit_2.SetParameters(0, 0)
line_fit_2.SetLineColor(ROOT.kGreen + 2)

gr_res_2 = ROOT.TGraphErrors(len(p_2), p_2, residuals_2, np.zeros_like(p_2), err_qeff_2)
gr_res_2.SetTitle("Residui in funzione di p (p > 400); p; q - q_{att}")
gr_res_2.SetMarkerStyle(20)
gr_res_2.SetMarkerColor(ROOT.kAzure + 2)
gr_res_2.SetLineColor(ROOT.kAzure + 3)

c_res_2 = ROOT.TCanvas("c_res_2", "Residui vs p 2", 800, 600)
c_res_2.SetGrid()
gr_res_2.Fit(line_fit_2, "S")
gr_res_2.Draw("AP")
c_res_2.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: hist_2 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_hist_2


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7.66353
NDf                       =           10
Edm                       =  5.83748e-07
NCalls                    =           71
Constant                  =      3.75115   +/-   1.19373     
Mean                      =     0.435569   +/-   1.78929     
Sigma                     =      7.38189   +/-   2.20361      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1952.27
NDf                       =           34
Edm                       =  2.67683e-08
NCalls                    =           33
p0                        =  0.000124611   +/-   6.21783e-05 
p1                        =   -0.0914187   +/-   0.0501251   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_res_2
